# Week 9 — Pandas Fundamentals

Cleaning the seeded trades extract: loading and inspecting it, detecting and fixing nulls and duplicates with justified methods, then reshaping and joining.

Input is `data/raw_trades.csv`, produced by `scripts/make_raw_trades.py`, which takes the 54-row BigQuery export and injects 6 null prices, 3 null volumes and 2 duplicate rows.

In [1]:
import numpy as np
import pandas as pd

## Monday — loading, dtypes and indexing

### 1. Load and inspect

Load `raw_trades.csv`, then report its shape, dtypes and null counts in three lines. *(Check: 56 rows.)*

In [2]:
# path is relative to this notebook, so it works on any clone of the repo
raw_trades = pd.read_csv('../data/raw_trades.csv')

In [3]:
raw_trades.head()

,trade_id,ticker,trade_date,price,volume,venue_code,side
0,1044,JNJ,2026-01-03,NaN,1000.0,XNYS,SELL
1,1045,JNJ,2026-01-03,NaN,7500.0,XNYS,SELL
2,1012,MSFT,2026-01-07,438.92,1500.0,XNAS,SELL
3,1002,AAPL,2026-01-08,224.96,1000.0,XNAS,BUY
4,1013,MSFT,2026-01-11,447.60,7500.0,XNAS,BUY


In [4]:
#show the shape of the dataframe (rows, columns)
raw_trades.shape

(56, 7)

In [5]:
#inspect the different data types
raw_trades.dtypes

trade_id        int64
ticker            str
trade_date        str
price         float64
volume        float64
venue_code        str
side              str
dtype: object

In [6]:
#count how many rows are null per column
raw_trades.isna().sum()

trade_id      0
ticker        0
trade_date    0
price         6
volume        3
venue_code    0
side          0
dtype: int64

#### Converting `trade_date` to a datetime

`to_datetime` is formally Thursday's topic, brought forward: converting *before* `set_index` gives a real `DatetimeIndex`, which lets the January selection below express intent directly and makes Wednesday's month grouping straightforward.

In [7]:
print(raw_trades['trade_date'].dtypes)

str


In [8]:
raw_trades['trade_date'] = pd.to_datetime(raw_trades['trade_date'])
raw_trades['trade_month'] = raw_trades['trade_date'].dt.month
raw_trades['trade_year'] = raw_trades['trade_date'].dt.year

In [9]:
raw_trades.dtypes

trade_id                int64
ticker                    str
trade_date     datetime64[us]
price                 float64
volume                float64
venue_code                str
side                      str
trade_month             int32
trade_year              int32
dtype: object

In [10]:
raw_trades.head()

,trade_id,ticker,trade_date,price,volume,venue_code,side,trade_month,trade_year
0,1044,JNJ,2026-01-03,NaN,1000.0,XNYS,SELL,1,2026
1,1045,JNJ,2026-01-03,NaN,7500.0,XNYS,SELL,1,2026
2,1012,MSFT,2026-01-07,438.92,1500.0,XNAS,SELL,1,2026
3,1002,AAPL,2026-01-08,224.96,1000.0,XNAS,BUY,1,2026
4,1013,MSFT,2026-01-11,447.60,7500.0,XNAS,BUY,1,2026


### 2. Index by date, select January two ways

Set `trade_date` as the index and select January's trades with both `.loc` and `.iloc`, then write one line on when each is appropriate.

In [11]:
#set trade date as index
raw_trades.set_index("trade_date", inplace=True)

In [12]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year
trade_date,,,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL,1,2026
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL,1,2026
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL,1,2026
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY,1,2026
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY,1,2026


In [13]:
raw_trades.sort_index(ascending=True, inplace=True)

In [14]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year
trade_date,,,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL,1,2026
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL,1,2026
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL,1,2026
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY,1,2026
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY,1,2026


In [15]:
#select january trades
#start and end
# raw_trades.loc['2026-01-01':'2026-01-31']
raw_trades.loc['2026-01']

,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year
trade_date,,,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL,1,2026
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL,1,2026
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL,1,2026
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY,1,2026
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY,1,2026
2026-01-13,1023,JPM,201.75,750.0,XNYS,BUY,1,2026
2026-01-15,1001,AAPL,230.02,2500.0,XNAS,SELL,1,2026
2026-01-17,1024,JPM,206.01,750.0,XNYS,SELL,1,2026
2026-01-27,1034,XOM,124.22,NaN,XNYS,BUY,1,2026


In [16]:
#positional
raw_trades.iloc[:9]

,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year
trade_date,,,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL,1,2026
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL,1,2026
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL,1,2026
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY,1,2026
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY,1,2026
2026-01-13,1023,JPM,201.75,750.0,XNYS,BUY,1,2026
2026-01-15,1001,AAPL,230.02,2500.0,XNAS,SELL,1,2026
2026-01-17,1024,JPM,206.01,750.0,XNYS,SELL,1,2026
2026-01-27,1034,XOM,124.22,NaN,XNYS,BUY,1,2026


**`.loc` vs `.iloc`:** use `.loc` when you mean the data itself — a date range like January — because labels survive sorting, filtering and row drops; use `.iloc` when you genuinely mean position rather than content, such as taking the top 5 after a deliberate sort, and accept that those positions shift the moment the frame changes.

### 3. The wrong dtype

**`trade_date` comes in as a string when it should be a datetime.** A CSV carries no type information at all — every value in the file is plain text. pandas infers types on read, and it can safely infer numbers, but dates are ambiguous (is `01/02/2026` 1 February or 2 January?), so it refuses to guess and leaves them as strings unless told explicitly via `parse_dates`. In BigQuery `trade_date` is declared `DATE` in the schema, so the type travels with the data; exporting to CSV throws that away.**Also worth noting: `volume` reads back as `float64`, not an integer**, despite share volumes being whole numbers. `NaN` is a float value and NumPy's `int64` has no representation for "missing", so a single null forces the entire column to upcast to float. Once the nulls are handled on Tuesday this can be converted back to `int`, or to pandas' nullable `Int64` which holds missing values while staying integer.

## Tuesday — cleaning

### 1. Detect the nulls

Confirm 6 nulls in `price` and 3 in `volume`, and show that `df.price == None` finds none of them.

In [17]:
raw_trades.isna().sum()

trade_id       0
ticker         0
price          6
volume         3
venue_code     0
side           0
trade_month    0
trade_year     0
dtype: int64

In [18]:
# .isna() finds the nulls; equality comparison finds none of them
print('found by .isna():   ', raw_trades.price.isna().sum())
print('found by == None:   ', (raw_trades.price == None).sum())
print('found by == np.nan: ', (raw_trades.price == np.nan).sum())

# why: NaN is not equal to anything, including itself
print()
print('np.nan == np.nan ->', np.nan == np.nan)
print('None == None     ->', None == None)

found by .isna():    6
found by == None:    0
found by == np.nan:  0

np.nan == np.nan -> False
None == None     -> True


**Why `== None` finds nothing.** Two separate reasons stack up:

1. **The missing values aren't `None`.** In a numeric column pandas stores missing as `NaN`, a float from the IEEE 754 spec — a different object entirely from Python's `None`. Comparing against `None` asks the wrong question.

2. **`NaN` isn't equal to anything, including itself.** IEEE 754 defines it that way: `np.nan == np.nan` is `False`. NaN means "undefined", and an undefined value can't be shown equal to anything — so even `== np.nan` returns all `False`.

The result is a mask that's `False` for all 56 rows, selecting nothing. `.isna()` exists precisely because equality can't do this job — it tests *is this value missing* rather than *does this value equal that one*.

**Contrast with SQL's `= NULL` (Week 8):** same practical outcome, different mechanism. SQL uses three-valued logic — `price = NULL` evaluates to `NULL` (unknown), and `WHERE` keeps only rows that are `TRUE`. pandas has no third value: the comparison returns plain `False`. Either way you need a purpose-built test — `IS NULL` in SQL, `.isna()` here.

### 2. Drop the duplicates

*(Check: 56 rows back down to 54.)*

In [19]:
#drop duplicates
raw_trades.drop_duplicates(inplace=True)

In [20]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year
trade_date,,,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL,1,2026
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL,1,2026
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL,1,2026
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY,1,2026
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY,1,2026


In [21]:
raw_trades.shape

(54, 8)

### 3. Fill the missing prices, per instrument

In [22]:
raw_trades['price'] = raw_trades['price'].fillna(raw_trades.groupby(['ticker'])['price'].transform('mean'))

In [23]:
raw_trades.isna().sum()

trade_id       0
ticker         0
price          0
volume         3
venue_code     0
side           0
trade_month    0
trade_year     0
dtype: int64

In [24]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year
trade_date,,,,,,,,
2026-01-03,1044,JNJ,152.345556,1000.0,XNYS,SELL,1,2026
2026-01-03,1045,JNJ,152.345556,7500.0,XNYS,SELL,1,2026
2026-01-07,1012,MSFT,438.920000,1500.0,XNAS,SELL,1,2026
2026-01-08,1002,AAPL,224.960000,1000.0,XNAS,BUY,1,2026
2026-01-11,1013,MSFT,447.600000,7500.0,XNAS,BUY,1,2026


#### Comparing a per-instrument fill against a global mean

Rebuilt from source below so both strategies start from the same 54 rows with the original 6 null prices intact. Overwriting the only copy of a column destroys the ability to compare alternatives, so each strategy is kept as its own named frame instead.

In [25]:
raw = (pd.read_csv('../data/raw_trades.csv')
         .set_index('trade_date')
         .sort_index()
         .drop_duplicates())

per_instrument = raw.assign(price=raw['price'].fillna(raw.groupby('ticker')['price'].transform('mean')))
global_mean    = raw.assign(price=raw['price'].fillna(raw['price'].mean()))

tickers = sorted(raw['ticker'].unique())
comparison = pd.DataFrame({
    'null_prices': raw[raw['price'].isna()]['ticker'].value_counts().reindex(tickers, fill_value=0),
    'per_instrument_fill': per_instrument.groupby('ticker')['price'].mean().round(2),
    'global_fill': global_mean.groupby('ticker')['price'].mean().round(2),
})
comparison['gap'] = (comparison['global_fill'] - comparison['per_instrument_fill']).round(2)

print(f"overall mean price, used by the global fill: {raw['price'].mean():.2f}\n")
comparison

overall mean price, used by the global fill: 226.46



,null_prices,per_instrument_fill,global_fill,gap
ticker,,,,
AAPL,1,221.88,222.30,0.42
JNJ,2,152.35,165.82,13.47
JPM,0,214.34,214.34,0.00
MSFT,3,468.00,402.12,-65.88
XOM,0,117.82,117.82,0.00


**Result.** The overall mean price across all five instruments is **£226.46**, and that single number is what a global fill would insert into every gap.

| ticker | null prices | per-instrument fill | global fill | gap |
|---|---|---|---|---|
| AAPL | 1 | 221.88 | 222.30 | +0.42 |
| JNJ | 2 | 152.35 | 165.82 | +13.47 |
| JPM | 0 | 214.34 | 214.34 | 0.00 |
| MSFT | 3 | 468.00 | 402.12 | **−65.88** |
| XOM | 0 | 117.82 | 117.82 | 0.00 |

**AAPL's gap is only £0.42** — much smaller than expected, and the reason is instructive rather than disappointing: AAPL caught just one of the six nulls, and it happens to trade at ~£222, almost exactly the overall mean of £226.46. A global fill inserts roughly the right number by luck. AAPL is the instrument this error is least visible on.

**MSFT is where the damage actually shows.** It took three of the six nulls and trades around £468, more than double the overall mean. Filling those three gaps with £226.46 drags MSFT's mean price down by **£65.88 — a 14% error** on an instrument whose price was never remotely near that number. JNJ is wrong in the opposite direction, pushed up £13.47 from £152.

**Why per-instrument is the defensible choice.** A global mean assumes every instrument is drawn from one distribution. These aren't — MSFT trades near £468, XOM near £118. The overall mean is a number no instrument in the basket actually trades at, so filling with it doesn't approximate any real price. Grouping by ticker fills each gap from that instrument's own price level, which is at least the right order of magnitude.

**A caveat worth recording even so.** A group mean uses *future* data to fill a past gap — MSFT's mean includes June trades, so a January null gets filled with information that didn't exist in January. For exploratory analysis that's acceptable. For a backtest or any "what did we know on 3 March" regulatory question it's lookahead bias, and it silently flatters results. A forward fill (carry the last observed price) would only use information already available at that point in time. This is Week 15's point-in-time correctness, and it's the reason this fill is fine *here* and would not be fine in a production price series.

#### The 3 missing volumes: flagged, not filled

**Volume is deliberately left unfilled.** Price has a defensible "roughly what was this instrument worth around then" answer; volume does not. A trade was either 2,500 shares or it wasn't — inventing a number manufactures an event that never happened. Volume also feeds straight into notional value (price × volume), VWAP and position sizing, so a fabricated volume becomes fabricated money downstream, and nothing marks it as invented.

A mean is doubly wrong here: it produces fractional shares, which aren't a thing.

What would be done in production, in order of preference:

1. **Go back to source.** Missing volume is usually an ingestion failure rather than a data failure — re-request the day from the vendor or reload the partition.
2. **Quarantine.** Route affected rows to a rejects table, let the clean rows flow, and alert someone. Nothing is silently dropped and nothing is silently invented.
3. **Flag and keep.** Where downstream genuinely needs the trade to exist, retain it with an explicit marker so consumers can exclude it from volume-weighted calculations.

Option 3 is taken below, since this is an analysis notebook rather than a pipeline. `volume` is also converted to pandas' nullable `Int64`, which restores integer semantics while still representing the missing values — resolving the float upcast noted on Monday.

In [26]:
# flag the missing volumes rather than inventing values for them
raw_trades['volume_is_missing'] = raw_trades['volume'].isna()

# nullable Int64 keeps whole-share semantics while still holding the missing values
raw_trades['volume'] = raw_trades['volume'].astype('Int64')

print(raw_trades.dtypes, '\n')
print(f"rows flagged: {raw_trades['volume_is_missing'].sum()}\n")
raw_trades[raw_trades['volume_is_missing']]

trade_id               int64
ticker                   str
price                float64
volume                 Int64
venue_code               str
side                     str
trade_month            int32
trade_year             int32
volume_is_missing       bool
dtype: object 

rows flagged: 3



,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year,volume_is_missing
trade_date,,,,,,,,,
2026-01-27,1034,XOM,124.22,<NA>,XNYS,BUY,1,2026,True
2026-03-13,1005,AAPL,209.18,<NA>,XNAS,SELL,3,2026,True
2026-03-18,1038,XOM,115.35,<NA>,XNYS,BUY,3,2026,True


## Wednesday — reshaping

### 1. `groupby` + `agg` with a dict

Sum of volume and mean of price in a single call. `volume` sums as a true integer here thanks to the nullable `Int64` conversion on Tuesday.

In [27]:
raw_trades.groupby(['ticker'])['price'].mean()

ticker
AAPL    221.882000
JNJ     152.345556
JPM     214.339091
MSFT    467.998750
XOM     117.824000
Name: price, dtype: float64

In [28]:
raw_trades.groupby(["ticker"]).agg({'price': 'mean', 'volume': 'sum'})

,price,volume
ticker,,
AAPL,221.882000,32250
JNJ,152.345556,54250
JPM,214.339091,16000
MSFT,467.998750,63000
XOM,117.824000,14750


### 2. Pivot table — total volume by ticker and month

*(Check: 5 × 6.)* `pivot` cannot do this: it only reshapes, and needs exactly one row per index/column pair. There are several trades per ticker per month, so the aggregation has to happen as part of the reshape — which is what `pivot_table` adds.

In [29]:
# min_count=1 so a group with no known volumes stays <NA> rather than summing to 0
pivot = raw_trades.pivot_table(columns='trade_month', index='ticker', values='volume',
                               aggfunc='sum', min_count=1)

In [30]:
pivot

trade_month,1,2,3,4,5,6
ticker,,,,,,
AAPL,3500,9500,2500,7500,8500,750
JNJ,8500,14000,1750,12500,7500,10000
JPM,1500,2500,6000,2250,2500,1250
MSFT,9000,4000,2000,9000,19500,19500
XOM,<NA>,7500,2000,1250,2500,1500


**Reconciling against Week 8's SQL.** The same figures were computed in BigQuery in Week 8, which makes this a genuine cross-check rather than a restatement — two independent implementations of the same question.

| month | BigQuery (volumes intact) | this pivot |
|---|---|---|
| 1 | 23,500 | 22,500 |
| 2 | 37,500 | 37,500 |
| 3 | 17,750 | 14,250 |
| 4 | 32,500 | 32,500 |
| 5 | 40,500 | 40,500 |
| 6 | 33,000 | 33,000 |

Four months agree exactly. January and March are short by precisely the volumes left unfilled on Tuesday — 1,000 in January (XOM), and 3,500 in March (AAPL 1,500 + XOM 2,000). The gap *is* the missing data, which is the correct behaviour: unknown inputs produce visibly reduced outputs rather than confident wrong ones.

**`min_count=1` matters more than it looks.** `aggfunc='sum'` skips nulls, so a group where *every* value is missing sums to `0` — XOM has exactly one January trade and its volume is unknown, so the cell claimed XOM traded zero shares that month. That is a fabricated fact, not a missing one. `min_count=1` requires at least one real value before returning a number, restoring `<NA>`.

Passing the builtin `sum` instead is wrong in the opposite direction: it discards *known* values that merely share a group with an unknown one, dropping March to 9,750 and losing AAPL's real 2,500 and XOM's real 2,000. Neither default announces itself — the reconciliation against BigQuery is the only reason any of this was visible.

### 3. Merge to attach sector

Merge the trades frame onto the instruments lookup (`data/instruments.csv`) to attach `sector`, then confirm the row count is still 54. If it grows, the join key is not unique — NVDA sits in that lookup with no trades against it.

In [31]:
instruments = pd.read_csv('../data/instruments.csv')

In [32]:
combined_data = pd.merge(raw_trades, instruments, on='ticker', how='inner')

In [33]:
combined_data.shape

(54, 12)

**Why the row count is the whole point.** 54 in, 54 out. That confirms two things at once: the join key is unique in the lookup (one row per ticker, so no trade matched more than once and fanned out), and the direction is right.

An `inner` join keeps only tickers present in both frames. NVDA sits in `instruments.csv` with no trades against it, so it is correctly absent — merging *from* the instruments side with a left join would have produced 55 rows, the extra one being NVDA carrying null trade fields. That is the same NVDA behaviour that made `INNER JOIN` return 54 rows and `LEFT JOIN` 55 in Week 8's SQL.

Checking the count after a join is a habit worth keeping: a silently inflated row count from a non-unique key is one of the easiest ways to double-count money in a pipeline, and nothing errors when it happens.

## Thursday — dates and rolling windows

### 1. Resample to month-end total volume

`resample` is `groupby` for a time index: it slices the `DatetimeIndex` into regular buckets and aggregates each one. Unlike grouping on a month integer it needs no helper column, keeps the year, labels buckets with real dates, and emits a row for every interval in the range — so a month with no trades shows as a gap rather than silently vanishing.

*(Check: should reproduce Wednesday's pivot column totals — 22,500 / 37,500 / 14,250 / 32,500 / 40,500 / 33,000.)*

In [34]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year,volume_is_missing
trade_date,,,,,,,,,
2026-01-03,1044,JNJ,152.345556,1000,XNYS,SELL,1,2026,False
2026-01-03,1045,JNJ,152.345556,7500,XNYS,SELL,1,2026,False
2026-01-07,1012,MSFT,438.920000,1500,XNAS,SELL,1,2026,False
2026-01-08,1002,AAPL,224.960000,1000,XNAS,BUY,1,2026,False
2026-01-11,1013,MSFT,447.600000,7500,XNAS,BUY,1,2026,False


In [35]:
trades_volume = raw_trades.resample('ME')['volume'].sum()

In [36]:
raw_trades.resample('ME').agg({'volume': 'sum', 'price': 'mean'}).round(2)

,volume,price
trade_date,,
2026-01-31,22500,242.02
2026-02-28,37500,239.85
2026-03-31,14250,203.52
2026-04-30,32500,236.62
2026-05-31,40500,246.99
2026-06-30,33000,253.28


### 2. Rolling 3-trade average price, per instrument

Compute a 3-trade rolling average price for each ticker using `groupby().transform()` — the same broadcasting behaviour used for Tuesday's fill, but with a rolling window inside rather than a plain mean.

In [37]:
raw_trades_new = raw_trades.copy()

In [38]:
raw_trades_new['rolling_3_trade_avg'] = raw_trades_new.groupby('ticker')['price'].transform(lambda x: x.rolling(3).mean().round(2))

In [39]:
raw_trades_new.head()

,trade_id,ticker,price,volume,venue_code,side,trade_month,trade_year,volume_is_missing,rolling_3_trade_avg
trade_date,,,,,,,,,,
2026-01-03,1044,JNJ,152.345556,1000,XNYS,SELL,1,2026,False,NaN
2026-01-03,1045,JNJ,152.345556,7500,XNYS,SELL,1,2026,False,NaN
2026-01-07,1012,MSFT,438.920000,1500,XNAS,SELL,1,2026,False,NaN
2026-01-08,1002,AAPL,224.960000,1000,XNAS,BUY,1,2026,False,NaN
2026-01-11,1013,MSFT,447.600000,7500,XNAS,BUY,1,2026,False,NaN


### 3. Prove the transform was per-group

The first 2 rows of *each* ticker should be null, not just the first 2 rows of the frame overall. A rolling window applied to the whole frame rather than per group would let one instrument's prices bleed into another's average — and would still return plausible-looking numbers. The null pattern is the proof.

In [40]:
raw_trades_new[raw_trades_new['rolling_3_trade_avg'].isna()]['ticker'].value_counts()

ticker
JNJ     2
MSFT    2
AAPL    2
JPM     2
XOM     2
Name: count, dtype: int64

**What this proves.** Ten nulls, split evenly as two per ticker across all five. That distribution is the evidence, not the total.

A rolling window applied to the frame as a whole would produce exactly **two** nulls — at the very first two rows — and every row after that would carry a number. Those numbers would look entirely reasonable: no error, no warning, nothing out of range. But AAPL's third row would be averaging itself against whichever JNJ and MSFT trades happened to precede it in date order, blending instruments that trade at £222 and £468 into a single meaningless figure.

Two nulls per group means each ticker's window started over from scratch, so no instrument's prices reached across into another's average. The null pattern is the only externally visible difference between the correct calculation and the plausible-looking wrong one — which is why the exercise asks for it rather than trusting the numbers to look right.

## Friday — apply vs transform vs agg, filtering, and first charts

### 1. The same aggregation three ways

Compute the same per-ticker aggregation using `apply`, `transform` and `agg`, then write one line on what each returns differently.

The distinction to draw out is **shape**: one collapses each group to a single row, one broadcasts a group-level value back across every original row, and one is the general-purpose escape hatch that can return either. Tuesday's fill needed the broadcasting kind and Wednesday's summary needed the collapsing kind — this is where that difference gets named.

In [41]:
raw_trades_new.groupby(['ticker'])['price'].agg('mean')

ticker
AAPL    221.882000
JNJ     152.345556
JPM     214.339091
MSFT    467.998750
XOM     117.824000
Name: price, dtype: float64

In [42]:
raw_trades_new.groupby('ticker').agg({'price': 'mean'})

,price
ticker,
AAPL,221.882000
JNJ,152.345556
JPM,214.339091
MSFT,467.998750
XOM,117.824000


In [43]:
raw_trades_new.groupby(['ticker'])['price'].transform(lambda x: x.mean())

trade_date
2026-01-03    152.345556
2026-01-03    152.345556
2026-01-07    467.998750
2026-01-08    221.882000
2026-01-11    467.998750
2026-01-13    214.339091
2026-01-15    221.882000
2026-01-17    214.339091
2026-01-27    117.824000
2026-02-02    467.998750
2026-02-06    117.824000
2026-02-06    152.345556
2026-02-09    152.345556
2026-02-09    221.882000
2026-02-12    117.824000
2026-02-13    214.339091
2026-02-14    467.998750
2026-02-16    221.882000
2026-03-03    214.339091
2026-03-03    152.345556
2026-03-06    117.824000
2026-03-13    221.882000
2026-03-15    214.339091
2026-03-18    467.998750
2026-03-18    117.824000
2026-03-19    152.345556
2026-03-26    221.882000
2026-04-04    117.824000
2026-04-08    214.339091
2026-04-11    152.345556
2026-04-18    221.882000
2026-04-21    117.824000
2026-04-22    467.998750
2026-04-23    152.345556
2026-04-24    214.339091
2026-04-26    467.998750
2026-05-02    467.998750
2026-05-05    214.339091
2026-05-13    221.882000
2026-05-18    

In [44]:
raw_trades_new.groupby('ticker')['price'].transform('mean')

trade_date
2026-01-03    152.345556
2026-01-03    152.345556
2026-01-07    467.998750
2026-01-08    221.882000
2026-01-11    467.998750
2026-01-13    214.339091
2026-01-15    221.882000
2026-01-17    214.339091
2026-01-27    117.824000
2026-02-02    467.998750
2026-02-06    117.824000
2026-02-06    152.345556
2026-02-09    152.345556
2026-02-09    221.882000
2026-02-12    117.824000
2026-02-13    214.339091
2026-02-14    467.998750
2026-02-16    221.882000
2026-03-03    214.339091
2026-03-03    152.345556
2026-03-06    117.824000
2026-03-13    221.882000
2026-03-15    214.339091
2026-03-18    467.998750
2026-03-18    117.824000
2026-03-19    152.345556
2026-03-26    221.882000
2026-04-04    117.824000
2026-04-08    214.339091
2026-04-11    152.345556
2026-04-18    221.882000
2026-04-21    117.824000
2026-04-22    467.998750
2026-04-23    152.345556
2026-04-24    214.339091
2026-04-26    467.998750
2026-05-02    467.998750
2026-05-05    214.339091
2026-05-13    221.882000
2026-05-18    

In [45]:
raw_trades_new.groupby('ticker')['price'].apply(lambda x: x.mean())

ticker
AAPL    221.882000
JNJ     152.345556
JPM     214.339091
MSFT    467.998750
XOM     117.824000
Name: price, dtype: float64

In [46]:
raw_trades_new.groupby('ticker')['price'].apply(lambda x: x - x.mean())

ticker  trade_date
AAPL    2026-01-08    3.078000e+00
        2026-01-15    8.138000e+00
        2026-02-09   -2.022000e+00
        2026-02-16   -6.022000e+00
        2026-03-13   -1.270200e+01
        2026-03-26   -6.432000e+00
        2026-04-18   -6.432000e+00
        2026-05-13   -2.842171e-14
        2026-05-20    4.818000e+00
        2026-06-03    2.768000e+00
        2026-06-18    1.480800e+01
JNJ     2026-01-03   -2.842171e-14
        2026-01-03   -2.842171e-14
        2026-02-06    4.694444e+00
        2026-02-09    7.034444e+00
        2026-03-03   -3.015556e+00
        2026-03-19   -9.435556e+00
        2026-04-11    1.614444e+00
        2026-04-23   -6.965556e+00
        2026-05-22    6.114444e+00
        2026-06-24   -2.815556e+00
        2026-06-25    2.774444e+00
JPM     2026-01-13   -1.258909e+01
        2026-01-17   -8.329091e+00
        2026-02-13   -3.269091e+00
        2026-03-03   -6.829091e+00
        2026-03-15   -5.429091e+00
        2026-04-08    1.280909e+00
 

**What each returns differently.** Same aggregation, same five numbers, three different shapes:

| call | rows out | index |
|---|---|---|
| `agg('mean')` | 5 | `ticker` — each group collapsed to one value |
| `transform('mean')` | 54 | `trade_date` — the group value broadcast back onto every original row |
| `apply(lambda x: x.mean())` | 5 | `ticker` — collapsed, because the function returned a scalar |
| `apply(lambda x: x - x.mean())` | 54 | `(ticker, trade_date)` — broadcast, because the function returned a Series |

`agg` always reduces and `transform` always broadcasts, but **`apply` does whichever the function leads it to** — the last two rows are the same method producing opposite shapes. That flexibility is why it is slower: it calls a Python function once per group, where the named aggregations dispatch to optimised implementations.

Two practical consequences:

**`apply` is not a drop-in for `transform` even when it broadcasts.** It prepends the group key, returning a `(ticker, trade_date)` MultiIndex rather than the original index, so assigning the result straight to a column will not align — the ticker level has to be dropped first. Tuesday's `fillna` worked because `transform` preserved the index exactly.

**The imputed rows are identifiable afterwards.** In the `x - x.mean()` output, AAPL on 2026-05-13 comes out at `-2.84e-14` — effectively zero. That is trade 1008, the AAPL row whose price was null and was filled with the group mean: subtract the mean from a value that *is* the mean and nothing is left. Filled values sit exactly on their group's centre and contribute no variance, which is one concrete way fabricated data distorts a distribution. That it reads `-2.84e-14` rather than a clean `0` is floating-point error — the same imprecision behind the `# TODO` on money handling in `src/trade_utils.py`.

### 2. Filter to the Technology tickers with `.isin()`

*(Check: 22 rows.)*

`.isin()` tests membership against a collection, which is how you express "any of these" without chaining `==` comparisons joined by `|`. Sector lives on the instruments lookup rather than the trades frame, so this needs the merged frame from Wednesday — or the ticker list read off the lookup.

The expected 22 should reconcile with Week 8's SQL, where counting trades per sector gave Technology 22, Healthcare 11, Financials 11, Energy 10.

In [47]:
combined_data[combined_data['sector'].isin(['Technology'])].info()

<class 'pandas.DataFrame'>
Index: 22 entries, 2 to 49
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   trade_id           22 non-null     int64  
 1   ticker             22 non-null     str    
 2   price              22 non-null     float64
 3   volume             21 non-null     Int64  
 4   venue_code         22 non-null     str    
 5   side               22 non-null     str    
 6   trade_month        22 non-null     int32  
 7   trade_year         22 non-null     int32  
 8   volume_is_missing  22 non-null     bool   
 9   instrument_id      22 non-null     int64  
 10  name               22 non-null     str    
 11  sector             22 non-null     str    
dtypes: Int64(1), bool(1), float64(1), int32(2), int64(2), str(5)
memory usage: 1.9 KB


### 3. Three charts

Straight off the DataFrame with `.plot()` — no plotting library imports needed beyond what pandas wraps:

1. a **line** of price over time,
2. a **bar** of volume per ticker,
3. a **histogram** of prices.

Enough to see the shape of the data without leaving the notebook. Worth noting what the histogram shows given five instruments at very different price levels — it is the same fact that made the global-mean fill indefensible on Tuesday, now visible rather than argued.